## Import libraries


In [ ]:
import json
import os
import pickle
import sys
import warnings

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

plt.rcParams["font.family"] = "DeJavu Serif"
plt.rcParams["font.serif"] = "Times New Roman"

out_dir = "/beegfs/halder/GITHUB/RESEARCH/crop-yield-forecasting-germany/output/figures"
leadtimes_dir = (
    "/beegfs/halder/GITHUB/RESEARCH/crop-yield-forecasting-germany/src/train/leadtimes"
)
scaler_dir = "/beegfs/halder/GITHUB/RESEARCH/crop-yield-forecasting-germany/src/scaler"

## Compute RMSE across all lead times for each crop and split


In [ ]:
def compute_rmse(predictions_scaled, targets_scaled, mean, std):
    """Inverse-scale predictions and targets, then compute RMSE (t/ha)."""
    preds = predictions_scaled * std + mean  # median quantile
    actuals = targets_scaled * std + mean
    return np.sqrt(np.mean((preds - actuals) ** 2))


# Crop configuration ─────────────────────────────────────────────────────────
crop_cfg = {
    "silage_maize": {
        "label": "Silage Maize",
        "scaler": "scaler_silage_maize.json",
        # full season = day 273; lead time = days_before_harvest
        "full_seq": 273,
    },
    "winter_wheat": {
        "label": "Winter Wheat",
        "scaler": "scaler_winter_wheat.json",
        "full_seq": 396,
    },
    "winter_barley": {
        "label": "Winter Barley",
        "scaler": "scaler_winter_barley.json",
        "full_seq": 426,
    },
}

# Collect results ─────────────────────────────────────────────────────────────
records = []

for crop, cfg in tqdm(crop_cfg.items()):
    # Load scaler
    with open(os.path.join(scaler_dir, cfg["scaler"]), "r") as f:
        scalers = json.load(f)
    y_mean = scalers["yield_mean"]
    y_std = scalers["yield_std"]

    crop_dir = os.path.join(leadtimes_dir, crop)
    day_folders = sorted(os.listdir(crop_dir), key=lambda x: int(x.split("_")[1]))

    for folder in tqdm(day_folders):
        day = int(folder.split("_")[1])
        lead_time_days = cfg["full_seq"] - day  # days before harvest
        folder_path = os.path.join(crop_dir, folder)

        # Load and concatenate validation + test outputs
        all_preds, all_targets = [], []
        for split in ("validation", "test"):
            pkl_path = os.path.join(folder_path, f"{split}_outputs.pkl")
            if not os.path.exists(pkl_path):
                continue
            with open(pkl_path, "rb") as f:
                outputs = pickle.load(f)
            # Column 1 is the median (q0.5)
            all_preds.append(outputs["prediction"][:, 1])
            all_targets.append(outputs["target"])

        if not all_preds:
            continue

        combined_preds = np.concatenate(all_preds)
        combined_targets = np.concatenate(all_targets)

        rmse = compute_rmse(combined_preds, combined_targets, y_mean, y_std)

        records.append(
            {
                "crop": crop,
                "crop_label": cfg["label"],
                "day": day,
                "lead_time_days": lead_time_days,
                "rmse": rmse,
            }
        )

results_df = pd.DataFrame(records)

csv_path = os.path.join(
    "/beegfs/halder/GITHUB/RESEARCH/crop-yield-forecasting-germany/output/tables",
    "forecast_skill_at_lead_times.csv",
)
os.makedirs(os.path.dirname(csv_path), exist_ok=True)
results_df.to_csv(csv_path, index=False, float_format="%.3f")

print(results_df.head(10))
print(f"\nShape: {results_df.shape}")

## Plot – RMSE vs. Lead Time

RMSE is computed on the **combined validation + test set** for each lead time and crop.


In [ ]:
crops_ordered = ["winter_wheat", "winter_barley", "silage_maize"]
crop_labels = {c: crop_cfg[c]["label"] for c in crops_ordered}
panel_labels = ["(a)", "(b)", "(c)"]

crop_colors = {
    "silage_maize": "#2b8cbe",
    "winter_wheat": "#e34a33",
    "winter_barley": "#31a354",
}

fig, axes = plt.subplots(
    1,
    3,
    figsize=(14, 4.5),
    dpi=300,
    sharey=False,
)

for ax, crop, panel in zip(axes, crops_ordered, panel_labels):
    crop_df = results_df[results_df["crop"] == crop].copy()
    crop_df = crop_df.sort_values("lead_time_days", ascending=False)

    ax.plot(
        crop_df["lead_time_days"],
        crop_df["rmse"],
        marker="o",
        color=crop_colors[crop],
        linewidth=1.8,
        markersize=7,
        markeredgecolor="k",
        markeredgewidth=0.5,
        zorder=3,
    )
    ax.fill_between(
        crop_df["lead_time_days"],
        crop_df["rmse"],
        crop_df["rmse"].min(),
        color=crop_colors[crop],
        alpha=0.10,
    )

    # Invert x-axis: long lead on left → harvest on right
    ax.invert_xaxis()

    ax.set_xlabel("Lead time (days before harvest)", fontsize=12)
    ax.set_ylabel("RMSE (t ha$^{-1}$)", fontsize=12)
    ax.set_title(crop_labels[crop], fontsize=12, fontweight="bold", pad=6)

    ax.text(
        0.98,
        0.97,
        panel,
        transform=ax.transAxes,
        fontsize=16,
        fontweight="bold",
        va="top",
        ha="right",
    )

    ax.tick_params(axis="both", labelsize=12)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True, nbins=6))
    ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.5)
    ax.set_facecolor("#fafafa")

plt.tight_layout()
plt.savefig(
    os.path.join(out_dir, "final", "forecast_skill_lead_times.svg"),
    bbox_inches="tight",
    dpi=300,
)
plt.savefig(
    os.path.join(out_dir, "final", "forecast_skill_lead_times.png"),
    bbox_inches="tight",
    dpi=300,
)
plt.show()
print("Figure saved.")